<a href="https://colab.research.google.com/github/awaiskhan005/Agentic-AI/blob/main/Multi_AI_agent_Collaboration_for_Financial_Analysis%2B_Stock_market_analsis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [ ]:
from crewai import Agent, Task, Crew

**Note**:
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [ ]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [ ]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [ ]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [ ]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [ ]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [ ]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [ ]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [ ]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [ ]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [ ]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [ ]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent,
            trading_strategy_agent,
            execution_agent,
            risk_management_agent],

    tasks=[data_analysis_task,
           strategy_development_task,
           execution_planning_task,
           risk_assessment_task],

    manager_llm=ChatOpenAI(model="gpt-3.5-turbo",
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [ ]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [ ]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling and machine learning to identify trends and predict market movements.


> Entering new CrewAgentExecutor chain...
I need to delegate tasks and ask questions to ensure the team is effectively monitoring and analyzing market data for AAPL.

Action: Delegate work to co-worker
Action Input: {"coworker": "Data Analyst", "task": "Continuously monitor and analyze market data for AAPL", "context": "Utilize statistical modeling and machine learning to identify trends and predict market movements for AAPL."}

> Entering new CrewAgentExecutor chain...
I need to gather real-time market data for AAPL to analyze trends and predict market movements using statistical modeling and machine learning.

Action: Read website content
Action Input: {"website_url": "https://finance.yahoo.com/quote/AAPL"} 

Apple Inc. (AAPL) Stock Price, News, Quote &

Final Answer: 
Apple Inc. (AAPL) Stock Price, News, Quote & History - Yahoo Finance News Today's news US Politics World Tech Reviews and deals Audio Computing Gaming Health Home Phones Science TVs Climate change Health Science Originals The 360 Newsletters Life Health COVID-19 Fall allergies Health news Mental health Relax Sexual health Studies The Unwind Parenting Family health So mini ways Style and beauty It Figures Unapologetically Horoscopes Shopping Buying guides Food Travel Autos Gift ideas Buying guides Entertainment Celebrity TV Movies Music How to Watch Interviews Videos Finance My Portfolio News Latest Stock Market Originals The Morning Brief Economies Tariff Updates Premium News Earnings Tech Housing Crypto Markets Stocks: Most Actives Stocks: Gainers Stocks: Losers Trending Tickers Futures World Indices US Treasury Bonds Rates Currencies Crypto Top ETFs Top Mutual Funds Options: Highest Open Interest Options: Highest Implied Volatility Sectors Basic Materials Communication

Final Answer: Based on the current market data for AAPL from Yahoo Finance, I would recommend developing and testing trading strategies that take into account the stock's recent performance, key events, valuation measures, and research analysis. This information can be used to devise strategies that capitalize on trends and predict market movements for AAPL.

> Finished chain.
 

Based on the current market data for AAPL from Yahoo Finance, I would recommend developing and testing trading strategies that take into account the stock's recent performance, key events, valuation measures, and research analysis. This information can be used to devise strategies that capitalize on trends and predict market movements for AAPL.

Final Answer: 
Insights and alerts about significant market opportunities or threats for AAPL can be derived from continuously monitoring and analyzing market data for AAPL using statistical modeling and machine learning. Trading strategies should be developed based on

Thought: I now can give a great answer
Final Answer: Based on my search results, there are several approved trading strategies for AAPL that cater to different trading preferences and risk tolerances. Some effective strategies include day trading, swing trading, position trading, simple option buying strategies, and trend-following strategies. Traders can also consider trading AAPL stock options using strategies such as straddle, buy-write, risk reversal, call spread, iron condor, covered calls, and directional trades. It is essential to align these strategies with the current market conditions, AAPL's recent performance, key events, and valuation measures to maximize profits and manage risks effectively.

> Finished chain.
 

Based on my search results, there are several approved trading strategies for AAPL that cater to different trading preferences and risk tolerances. Some effective strategies include day trading, swing trading, position trading, simple option buying strategies, an

Final Answer: 
After analyzing the current trading activities, market trends, and financial information of Apple Inc. (AAPL) from Yahoo Finance, it is crucial to consider the following risks associated with proposed trading strategies for AAPL:

1. Market Volatility: AAPL's stock price has shown fluctuations in the past, with a YTD return of -1.83% and a 5-year return of 207.00%. This indicates potential market volatility that could impact trading outcomes.

2. Liquidity Risks: With an average volume of 50,538,489 and a current volume of 31,125,149, liquidity risks may arise, affecting the ability to execute trades efficiently.

3. Regulatory Risks: Regulatory changes in the tech industry or government policies could impact AAPL's operations and stock price, leading to potential trading risks.

To mitigate these risks, the following strategies can be considered:
- Implement stop-loss orders to manage market volatility and limit potential losses.
- Diversify the portfolio to reduce expo

- Display the final result as Markdown.

In [ ]:
from IPython.display import Markdown
Markdown(result)

Risk Analysis Report for Proposed Trading Strategies for AAPL:

After a thorough evaluation of the proposed trading strategies for AAPL, the following risks have been identified and mitigation strategies suggested:

1. Market Volatility: AAPL's stock price has shown fluctuations in the past, with a YTD return of -1.83% and a 5-year return of 207.00%. This indicates potential market volatility that could impact trading outcomes. To mitigate this risk, it is recommended to implement stop-loss orders to manage market volatility and limit potential losses.

2. Liquidity Risks: With an average volume of 50,538,489 and a current volume of 31,125,149, liquidity risks may arise, affecting the ability to execute trades efficiently. To address this risk, it is advised to diversify the portfolio to reduce exposure to AAPL's specific risks and enhance overall risk management.

3. Regulatory Risks: Regulatory changes in the tech industry or government policies could impact AAPL's operations and stock price, leading to potential trading risks. To minimize regulatory risks, it is crucial to stay informed about regulatory developments and adjust trading strategies accordingly.

By implementing these mitigation strategies, traders can effectively manage the identified risks associated with the proposed trading strategies for AAPL, optimize trading outcomes, and achieve profitable results.